# Semana 5 · Del contrato Wine al formulario

`->` Quiero trasladar a Wine Quality el flujo que vi en los ejemplos de Churn. Primero compruebo los campos y la muestra. Después compruebo cuándo llama la interfaz al gateway.

`~>` Reutilizo el [esquema del starter](../modules/05-streamlit-basic-model-ui/exercises/02-first-streamlit/problem/starter/src/model_ui/ui_schema.py) que incluye la versión del repo. Cada `FieldSpec` separa la etiqueta que ve el usuario del nombre que espera S4. Los rangos vienen del contrato, los valores iniciales de la primera muestra de S3 y los pasos se ajustan a la precisión del campo. Por ejemplo, uso 0,1 para alcohol y 0,0001 para densidad.

Abro este notebook con el entorno Python del starter de Wine de S5, preparado con `uv sync --locked`. El directorio de trabajo del kernel debe estar dentro del repo.

In [1]:
import csv
import json
import sys
from pathlib import Path

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents) if (path / "zip_semana5").is_dir())
starter = root / (
    "zip_semana5/modules/05-streamlit-basic-model-ui/"
    "exercises/02-first-streamlit/problem/starter"
)
source_s4 = root / (
    "semana4/modules/04-model-packaging/exercises/01-model-packaging/"
    "01.02-serializable-inference-module/problem/starter/src"
)
sys.path.insert(0, str(starter / "src"))
sys.path.insert(0, str(source_s4))

from model_packaging.contracts import WineQualityRequest
from model_ui.ui_schema import FIELD_SPECS

manifest = root / "semana4/assets/04-model-packaging/manifest_example.json"
manifest_names = tuple(json.loads(manifest.read_text(encoding="utf-8"))["feature_names"])
properties = WineQualityRequest.model_json_schema()["properties"]
names = tuple(spec.name for spec in FIELD_SPECS)

assert len(names) == len(set(names)) == 11
assert names == manifest_names == tuple(properties)

for spec in FIELD_SPECS:
    limits = properties[spec.name]
    assert (spec.min_value, spec.max_value) == (limits["minimum"], limits["maximum"])
    assert spec.min_value <= spec.default <= spec.max_value and spec.step > 0
    assert spec.label.strip() and spec.label != spec.name
    print(f"{spec.name:24} {spec.label} · {spec.min_value:g}–{spec.max_value:g}"
          f" · inicial {spec.default:g} · paso {spec.step:g}")

fixed_acidity            Acidez fija · 0–20 · inicial 7.4 · paso 0.1
volatile_acidity         Acidez volátil · 0–2 · inicial 0.7 · paso 0.01
citric_acid              Ácido cítrico · 0–2 · inicial 0 · paso 0.01
residual_sugar           Azúcar residual · 0–20 · inicial 1.9 · paso 0.1
chlorides                Cloruros · 0–1 · inicial 0.076 · paso 0.001
free_sulfur_dioxide      Dióxido de azufre libre · 0–100 · inicial 11 · paso 1
total_sulfur_dioxide     Dióxido de azufre total · 0–300 · inicial 34 · paso 1
density                  Densidad · 0.98–1.01 · inicial 0.9978 · paso 0.0001
ph                       pH · 2.5–4.5 · inicial 3.51 · paso 0.01
sulphates                Sulfatos · 0–3 · inicial 0.56 · paso 0.01
alcohol                  Alcohol · 5–20 · inicial 9.4 · paso 0.1


`>` Construyo el payload de `red-001` con la [muestra de S3](../../semana3/assets/03-wine-quality/inference_samples.csv). Conservo las claves y el orden del manifiesto. Las etiquetas traducidas y `sample_id` se quedan fuera de la petición.

In [2]:
samples = root / "semana3/assets/03-wine-quality/inference_samples.csv"
with samples.open(encoding="utf-8", newline="") as csv_file:
    sample = next(csv.DictReader(csv_file))

payload = {spec.name: float(sample[spec.name]) for spec in FIELD_SPECS}
assert tuple(payload) == manifest_names
assert payload == {spec.name: spec.default for spec in FIELD_SPECS}
request = WineQualityRequest.model_validate(payload)
print(sample["sample_id"])
print(json.dumps(request.model_dump(), indent=2))

red-001
{
  "fixed_acidity": 7.4,
  "volatile_acidity": 0.7,
  "citric_acid": 0.0,
  "residual_sugar": 1.9,
  "chlorides": 0.076,
  "free_sulfur_dioxide": 11.0,
  "total_sulfur_dioxide": 34.0,
  "density": 0.9978,
  "ph": 3.51,
  "sulphates": 0.56,
  "alcohol": 9.4
}


`>` Distingo la edición del envío con `st.form`, como en los ejemplos de Churn del guion de clase.

| Acción | Qué ocurre | Llamadas al gateway |
| --- | --- | ---: |
| Edito un campo del formulario | El navegador conserva el valor pendiente. La app todavía no recibe un submit. | 0 |
| Pulso **Ejecutar inferencia** | Streamlit envía los valores juntos y reejecuta el script. El botón devuelve `True` y la app pasa el payload al gateway. | 1 |

`>>` Ejecuto los dos tests docentes de conteo y los de presentación y error. Los dobles pertenecen a los tests. El QA de la [app con el bundle real](evidencias/5.2-streamlit.txt) está registrado por separado.

In [3]:
from subprocess import run

checks = (
    "no_submit_makes_zero_gateway_calls or "
    "submit_makes_exactly_one_gateway_call or "
    "render_prediction_shows_all_four_payload_fields or "
    "inference_bundle_error_is_actionable_and_hides_details or "
    "invalid_inference_error_is_actionable_and_hides_details"
)
result = run(
    ["uv", "run", "--locked", "python", "-m", "pytest", "-q",
     "tests/test_app.py", "-k", checks],
    cwd=starter,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
result.check_returncode()

......                                                                   [100%]
6 passed, 8 deselected in 0.11s


`~>` En éxito, muestro categoría y confianza primero, con las versiones de modelo y preprocesado como datos de trazabilidad. La confianza es una señal del modelo. Si el bundle falta, indico que hay que configurar `MODEL_UI_BUNDLE` y relanzar. Si falla la inferencia, pido revisar la muestra y la compatibilidad del bundle. Los mensajes mantienen los detalles técnicos fuera de la pantalla.

`=>` El notebook confirma las once claves, sus rangos, la muestra validada por S4 y el contrato de una llamada por submit. Paso a S6 con el mismo esquema. Allí incorporo el último resultado, los estados visibles y el reintento.